# CSC 4792 Mini Project — Chililabombwe Municipal Council Dataset

**Course:** CSC 4792 — Data Mining and Warehousing  
**Institution:** University of Zambia  
**Academic Year:** 2025/26  
**Project Team:** #11  
**Assigned Council:** Chililabombwe Municipal Council  
**Website:** http://www.chililabombwecouncil.gov.zm  

**Team Members:**
- Isaiah Chileshe
- Inkumbu Chawewa
- Bright Chingwala
- Faith Nswana
- Karen Chipampe

---

## 📋 Project Overview

This notebook documents the complete pipeline for creating a multi-source 
dataset of Chililabombwe Municipal Council's budgets, Constituency 
Development Fund (CDF) projects, and 2025 output targets. The dataset is 
published on Kaggle and the notebook is version-controlled on GitHub.

**Dataset link:**  
https://www.kaggle.com/datasets/isaiahchileshe/chililabombwe-municipal-council-data-zambia

**GitHub repository:**  
https://github.com/chileshe-dev/db-unza26-csc4792-chililabombwe

---

## 🎯 Objectives

1. Scrape the council's official website for relevant PDF documents
2. Extract structured data from text-based and scanned PDFs
3. Clean, normalise, and validate the extracted data
4. Export pipe-separated CSVs (`db-unza26-csc4792-*.csv`)
5. Publish the dataset on Kaggle with full documentation

---

## 📁 Deliverables Produced by This Notebook

| # | File | Rows | Content |
|---|---|---|---|
| 1 | `db-unza26-csc4792-chililabombwe_obb_budget_2024.csv` | 60 | 2024 Output-Based Budget |
| 2 | `db-unza26-csc4792-chililabombwe_budget_comparison_2023_2024.csv` | 4 | YoY budget growth |
| 3 | `db-unza26-csc4792-chililabombwe_budget_programmes_2025.csv` | 59 | 3-year programme comparison |
| 4 | `db-unza26-csc4792-chililabombwe_cdf_projects_2024.csv` | 13 | Approved CDF projects |
| 5 | `db-unza26-csc4792-chililabombwe_output_targets_2025.csv` | 11 | 2025 output KPIs |
| 6 | `db-unza26-csc4792-chililabombwe_revenue_sources_2025.csv` | 23 | Revenue by source |

---

## 🛠️ Methodology

### Phase 1 — Site reconnaissance
- Mapped the council website structure using `requests` + `BeautifulSoup`
- Identified two data-rich pages: *Publications* and *Council Documents*

### Phase 2 — Document harvesting
- Automatically downloaded **46 PDF files** (~180 MB) covering budgets, 
  financial statements, council minutes, and IDPs

### Phase 3 — Document classification
- Diagnostic scan with `pdfplumber` split documents into:
  - **Text-based PDFs** (25 files) — extractable with `extract_tables()`
  - **Scanned image PDFs** (21 files) — required manual transcription

### Phase 4 — Data extraction
- **Text PDFs:** `pdfplumber` + custom regex parsing
- **Scanned PDFs:** manual transcription from 300 DPI PNGs (`PyMuPDF`)

### Phase 5 — Cleaning & validation
- Normalised encoding artefacts, fixed whitespace, corrected item numbering
- **Validated budget totals against printed totals in source PDFs**:
  - 2024 OBB: K147,958,491 (exact match ✅)
  - 2025 programme total: K222,580,695 (99.5% match ✅)

### Phase 6 — Export
- All files exported as **pipe-separated (`|`) CSVs** per assignment spec

---



## 📚 References

- Local Government Act, 2019 (Act No. 2 of 2019). Republic of Zambia.
- The Constituency Development Fund Act No. 11 of 2018. Republic of Zambia.
- Phiri, L. (2026). *A Multi-Source Dataset for CS1 Failure Prediction* 
  [Dataset]. Kaggle. https://www.kaggle.com/lightonphiri/a-multisource-dataset-for-cs1-failure-prediction

---

*Last updated: September 2026*

In [1]:
# ================================
# NOTEBOOK SETUP — we run this cell
# first after any kernel restart
# ================================
import os
import re
import time
import requests
import urllib3
import pandas as pd
import pdfplumber
from bs4 import BeautifulSoup
from urllib.parse import urlparse

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

def safe_print(text, limit=500):
    s = str(text)
    print(s[:limit])
    if len(s) > limit:
        print(f"... [{len(s) - limit} more characters truncated]")

print("✅ Setup complete")

✅ Setup complete


In [ ]:
# Chililabombwe Municipal Council — Data Scraping Notebook

**Course:** CSC 4792 — Data Mining and Warehousing  
**Project Team:** #11  
**Assigned Council:** Chililabombwe Municipal Council  
**Website:** http://www.chililabombwecouncil.gov.zm  
**Date started:** 2026-09-11

## Objectives
1. Scrape council data (CDF projects, budgets, revenue, IDPs)
2. Clean and structure the data
3. Export as pipe-separated CSVs (`db-unza26-csc4792-*.csv`)

## Site Accessibility Notes
- The council website is served over HTTP; browsers flag it as "Not Secure."
- HTTPS requests fail due to missing/invalid SSL certificate.
- Scraping will use `http://` with `verify=False`.

In [4]:
import requests
from bs4 import BeautifulSoup
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://www.chililabombwecouncil.gov.zm"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

try:
    response = requests.get(url, headers=headers, timeout=30, verify=False, allow_redirects=True)
    print("Status code:", response.status_code)
    print("Final URL:", response.url)
    print("Page length:", len(response.text))
    soup = BeautifulSoup(response.text, "html.parser")
    print("Title:", soup.title.string if soup.title else "No title")
except Exception as e:
    print("ERROR:", e)

Status code: 200
Final URL: https://www.chililabombwecouncil.gov.zm/
Page length: 109288
Title: Chililabombwe Municipal Council – Chililabombwe


## Initial Connectivity Test — 2026-09-11

- Request to `http://www.chililabombwecouncil.gov.zm` returned status **200**.
- Site redirects to `https://www.chililabombwecouncil.gov.zm/`.
- Final page fetched successfully; HTML length ≈ 109 KB.
- Title confirms council identity: "Chililabombwe Municipal Council - Chililabombwe".
- **Decision:** Use HTTPS endpoint for subsequent scraping.

## Step 1: Website Reconnaissance

**Goal:** Map the council website to identify pages that contain useful data.

**Method:** Use `requests` to fetch the homepage and `BeautifulSoup` to 
enumerate all links.

**Expected output:** List of all navigation links with their URLs.

In [5]:
# Extract all links from the homepage
soup = BeautifulSoup(response.text, "html.parser")

links = []
for a in soup.find_all("a", href=True):
    text = a.get_text(strip=True)
    href = a["href"]
    if text:  # skip empty link text
        links.append((text, href))

# Show all unique links with their text
for text, href in links:
    print(f"{text[:60]:60} -> {href}")

Home                                                         -> https://www.chililabombwecouncil.gov.zm/
About                                                        -> #
Our District                                                 -> #
About Us                                                     -> https://www.chililabombwecouncil.gov.zm/?page_id=2937
Places to Visit and Relax                                    -> https://www.chililabombwecouncil.gov.zm/?page_id=3009
Mandate                                                      -> https://www.chililabombwecouncil.gov.zm/?page_id=169
Who we are                                                   -> https://www.chililabombwecouncil.gov.zm/?page_id=118
Departments                                                  -> https://www.chililabombwecouncil.gov.zm/?page_id=770
Services                                                     -> https://www.chililabombwecouncil.gov.zm/?page_id=792
News Updates                                               

In [6]:
# Try to isolate the main navigation menu
nav = soup.find("nav")
if nav:
    print("NAV FOUND:")
    for a in nav.find_all("a", href=True):
        print(" -", a.get_text(strip=True), "->", a["href"])
else:
    print("No <nav> tag found — dumping all <a> tags instead")

# Also check for common menu class names
for cls in ["menu", "navbar", "main-menu", "navigation"]:
    found = soup.find_all(class_=lambda x: x and cls in x.lower())
    if found:
        print(f"\nElements with class containing '{cls}': {len(found)}")

NAV FOUND:
 - Home -> https://www.chililabombwecouncil.gov.zm/
 - About -> #
 - Our District -> #
 - About Us -> https://www.chililabombwecouncil.gov.zm/?page_id=2937
 - Places to Visit and Relax -> https://www.chililabombwecouncil.gov.zm/?page_id=3009
 - Mandate -> https://www.chililabombwecouncil.gov.zm/?page_id=169
 - Who we are -> https://www.chililabombwecouncil.gov.zm/?page_id=118
 - Departments -> https://www.chililabombwecouncil.gov.zm/?page_id=770
 - Services -> https://www.chililabombwecouncil.gov.zm/?page_id=792
 - News Updates -> https://www.chililabombwecouncil.gov.zm/?page_id=187
 - ZDSP -> https://www.chililabombwecouncil.gov.zm/?page_id=1909
 - About ZDSP -> https://www.chililabombwecouncil.gov.zm/?page_id=3116
 - Media Center -> #
 - Publications -> https://www.chililabombwecouncil.gov.zm/?page_id=959
 - Photo Gallery -> https://www.chililabombwecouncil.gov.zm/?page_id=971
 - Council Documents -> https://www.chililabombwecouncil.gov.zm/?page_id=2463
 - Contact Us -> ht

In [7]:
import time

def explore_page(url, label=""):
    """Fetch a page and list all links, PDFs, and tables found on it."""
    print(f"\n{'='*70}")
    print(f"EXPLORING: {label} — {url}")
    print('='*70)
    
    try:
        r = requests.get(url, headers=headers, timeout=30, verify=False)
        print(f"Status: {r.status_code} | Length: {len(r.text)}")
        s = BeautifulSoup(r.text, "html.parser")
        
        # Page title
        if s.title:
            print(f"Title: {s.title.get_text(strip=True)}")
        
        # All links
        links = [(a.get_text(strip=True), a["href"]) for a in s.find_all("a", href=True)]
        print(f"\n--- {len(links)} LINKS FOUND ---")
        for text, href in links:
            if text:
                print(f"  {text[:70]:70} -> {href}")
        
        # All tables
        tables = s.find_all("table")
        print(f"\n--- {len(tables)} TABLES FOUND ---")
        for i, t in enumerate(tables):
            rows = t.find_all("tr")
            print(f"  Table {i+1}: {len(rows)} rows")
        
        # All images (may include posters/scans of data)
        images = s.find_all("img", src=True)
        print(f"\n--- {len(images)} IMAGES FOUND ---")
        
    except Exception as e:
        print(f"ERROR: {e}")
    
    time.sleep(2)  

In [8]:
import time

def explore_page(url, label=""):
    """Fetch a page and list all links, PDFs, and tables found on it."""
    print(f"\n{'='*70}")
    print(f"EXPLORING: {label} — {url}")
    print('='*70)
    
    try:
        r = requests.get(url, headers=headers, timeout=30, verify=False)
        print(f"Status: {r.status_code} | Length: {len(r.text)}")
        s = BeautifulSoup(r.text, "html.parser")
        
        # Page title
        if s.title:
            print(f"Title: {s.title.get_text(strip=True)}")
        
        # All links
        links = [(a.get_text(strip=True), a["href"]) for a in s.find_all("a", href=True)]
        print(f"\n--- {len(links)} LINKS FOUND ---")
        for text, href in links:
            if text:
                print(f"  {text[:70]:70} -> {href}")
        
        # All tables
        tables = s.find_all("table")
        print(f"\n--- {len(tables)} TABLES FOUND ---")
        for i, t in enumerate(tables):
            rows = t.find_all("tr")
            print(f"  Table {i+1}: {len(rows)} rows")
        
        # All images (may include posters/scans of data)
        images = s.find_all("img", src=True)
        print(f"\n--- {len(images)} IMAGES FOUND ---")
        
    except Exception as e:
        print(f"ERROR: {e}")
    
    time.sleep(2)  

In [9]:
# Explore the most important pages
explore_page("https://www.chililabombwecouncil.gov.zm/?page_id=2449", "CDF Page")
explore_page("https://www.chililabombwecouncil.gov.zm/?page_id=2463", "Council Documents")
explore_page("https://www.chililabombwecouncil.gov.zm/?page_id=959", "Publications")
explore_page("https://www.chililabombwecouncil.gov.zm/?page_id=1909", "ZDSP")
explore_page("https://www.chililabombwecouncil.gov.zm/?page_id=932", "View Tracker")


EXPLORING: CDF Page — https://www.chililabombwecouncil.gov.zm/?page_id=2449
Status: 200 | Length: 76752
Title: CDF – Chililabombwe Municipal Council

--- 29 LINKS FOUND ---
  Home                                                                   -> https://www.chililabombwecouncil.gov.zm/
  About                                                                  -> #
  Our District                                                           -> #
  About Us                                                               -> https://www.chililabombwecouncil.gov.zm/?page_id=2937
  Places to Visit and Relax                                              -> https://www.chililabombwecouncil.gov.zm/?page_id=3009
  Mandate                                                                -> https://www.chililabombwecouncil.gov.zm/?page_id=169
  Who we are                                                             -> https://www.chililabombwecouncil.gov.zm/?page_id=118
  Departments                      

In [10]:
import os
import time
from urllib.parse import urlparse

# Create a folder for PDFs
os.makedirs("pdfs", exist_ok=True)

def download_pdfs(page_url, save_dir="pdfs"):
    """Scrape a page, download every PDF link found on it."""
    r = requests.get(page_url, headers=headers, timeout=30, verify=False)
    s = BeautifulSoup(r.text, "html.parser")
    
    pdf_links = []
    for a in s.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".pdf"):
            # Handle both absolute and relative URLs
            if href.startswith("http"):
                full_url = href
            else:
                full_url = "https://www.chililabombwecouncil.gov.zm/" + href.lstrip("/")
            text = a.get_text(strip=True)
            pdf_links.append((text, full_url))
    
    print(f"Found {len(pdf_links)} PDFs on {page_url}\n")
    
    for text, url in pdf_links:
        filename = os.path.basename(urlparse(url).path)
        filepath = os.path.join(save_dir, filename)
        
        if os.path.exists(filepath):
            print(f"[SKIP] {filename} (already downloaded)")
            continue
        
        try:
            pdf_r = requests.get(url, headers=headers, timeout=60, verify=False)
            if pdf_r.status_code == 200:
                with open(filepath, "wb") as f:
                    f.write(pdf_r.content)
                size_kb = len(pdf_r.content) / 1024
                print(f"[OK] {filename} ({size_kb:.0f} KB)")
            else:
                print(f"[FAIL {pdf_r.status_code}] {filename}")
        except Exception as e:
            print(f"[ERROR] {filename}: {e}")
        
        time.sleep(1)  

# Download everything from Publications (main data page)
download_pdfs("https://www.chililabombwecouncil.gov.zm/?page_id=959")

Found 42 PDFs on https://www.chililabombwecouncil.gov.zm/?page_id=959

[SKIP] 2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf (already downloaded)
[SKIP] 2024-2026-OUTPUT-BASED-BUDGET-CHILILABOMBWE-HEAD-MANDATE-OBJECTIVES-WITH-SUB-PROGRAMMES.pdf (already downloaded)
[SKIP] Bi-Annual-Perormance-Report-2024.pdf (already downloaded)
[SKIP] Annual-Performance-Report-2024.pdf (already downloaded)
[SKIP] Performance-Analysis-2023.pdf (already downloaded)
[SKIP] Bi-Annual-Perormance-Report-2025.pdf (already downloaded)
[SKIP] 2025-CHILILABOMBWE-MUNICIPAL-COUNCIL-ADJUSTED-BUDGET.pdf (already downloaded)
[SKIP] CHILILABOMBWE-MUNICIPAL-COUNCIL-2026-APPROVED-BUDGET.pdf (already downloaded)
[SKIP] Financial-statement-2023.pdf (already downloaded)
[SKIP] Financial-statements-2022.pdf (already downloaded)
[SKIP] Audit-opinion-for-the-year-2022-financial-statement.pdf (already downloaded)
[SKIP] Financial-statements-2021.pdf (already downloaded)
[SKIP] Financial-statements-2020.pdf (already 

In [11]:
!pip install pdfplumber pandas openpyxl

In [12]:
import pdfplumber
import pandas as pd


In [13]:
import os

pdf_files = sorted([f for f in os.listdir("pdfs") if f.endswith(".pdf")])
print(f"Total PDFs in pdfs/ folder: {len(pdf_files)}\n")
for f in pdf_files:
    size_kb = os.path.getsize(f"pdfs/{f}") / 1024
    print(f"  {size_kb:8.1f} KB  {f}")

Total PDFs in pdfs/ folder: 41

     480.8 KB  12-TH-ORDINARY-COUNCIL-28-JUNE-2024.pdf
    1074.3 KB  14TH-ORDINARY-COUNCIL-2024.pdf
      29.1 KB  2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf
     680.7 KB  2024-2026-OUTPUT-BASED-BUDGET-CHILILABOMBWE-HEAD-MANDATE-OBJECTIVES-WITH-SUB-PROGRAMMES.pdf
     741.1 KB  2024-Actionable-stakeholder-engagement-plan-1.pdf
     761.5 KB  2025-CHILILABOMBWE-MUNICIPAL-COUNCIL-ADJUSTED-BUDGET.pdf
     767.7 KB  2025-Community-projects-applications.pdf
      82.4 KB  2025-approved-Community-projects.pdf
     149.9 KB  Act-No.-3-The-Data-Protection-Act-2021_0-2.pdf
    3149.5 KB  Annual-Performance-Report-2024.pdf
     137.4 KB  Audit-opinion-for-the-year-2022-financial-statement.pdf
    4205.5 KB  Bi-Annual-Perormance-Report-2024.pdf
    2876.3 KB  Bi-Annual-Perormance-Report-2025.pdf
    6346.2 KB  CHILILABOMBWE-MUNICIPAL-COUNCIL-2026-APPROVED-BUDGET.pdf
    5908.1 KB  CITZEN-ENGAMENT-STRATEGY-1.pdf
     869.0 KB  CONYENACE-NOTIFICATION-

## PDF Harvest — 2026-09-11

42 PDF documents were downloaded from the Publications page
(?page_id=959) into a local `pdfs/` directory.

### Document Categories & Priority for Extraction

**Tier 1 — Direct Data (extract tables immediately):**
- `List-of-CDF-approved-projects-for-2024.pdf`
- `2025-approved-Community-projects.pdf`
- `CHILILABOMBWE-MUNICIPAL-COUNCIL-2026-APPROVED-BUDGET.pdf`
- `2025-CHILILABOMBWE-MUNICIPAL-COUNCIL-ADJUSTED-BUDGET.pdf`
- `2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf`
- `2024-2026-OUTPUT-BASED-BUDGET-...-SUB-PROGRAMMES.pdf`
- `Bi-Annual-Perormance-Report-2024.pdf` / `2025.pdf`
- `Annual-Performance-Report-2024.pdf`
- `Performance-Analysis-2023.pdf`
- `Financial-statement-2018...2024.pdf` (7 years)

**Tier 2 — Council Resolutions & Minutes:**
- Minutes of Ordinary Council Meetings (11th–14th, 2024; 1st–2nd, 2025)
- Budget stakeholder meeting minutes
- Community engagement minutes

**Tier 3 — Contextual / Legal:**
- CDF Act 2018, Public Finance Act 2018, ICT Policy, etc.

Total disk footprint: ~180 MB

In [14]:
def safe_print(text, limit=500):
    """Print only the first `limit` characters of `text`."""
    s = str(text)
    print(s[:limit])
    if len(s) > limit:
        print(f"... [{len(s) - limit} more characters truncated]")

In [15]:
download_pdfs("https://www.chililabombwecouncil.gov.zm/?page_id=2463")

Found 5 PDFs on https://www.chililabombwecouncil.gov.zm/?page_id=2463

[OK] CHILILABOMBWE-DISTRICT-IDP-2022-2031.pdf (7821 KB)
[OK] Service-charter-cmc-2023.pdf (1083 KB)
[OK] CHILILABOMBWE-MUNICIPAL-ICT-STRATEGIC-PLAN.pdf (1030 KB)
[OK] Chililabombwe-Municipal-ICT-Policy-and-Procedure-Manual.pdf (728 KB)
[OK] List-of-CDF-approved-projects-for-2024.pdf (488 KB)


In [16]:
import os

# 1. Confirm functions are defined
print("download_pdfs defined:", "download_pdfs" in dir())
print("safe_print defined:", "safe_print" in dir())
print()

# 2. Confirm PDFs are there
if os.path.exists("pdfs"):
    pdfs = [f for f in os.listdir("pdfs") if f.lower().endswith(".pdf")]
    print(f"PDFs in folder: {len(pdfs)}")
    # Show the CDF-related ones specifically
    print("\nCDF / Community / Project PDFs found:")
    for f in pdfs:
        if any(k in f.lower() for k in ["cdf", "community", "project", "ward"]):
            size_kb = os.path.getsize(f"pdfs/{f}") / 1024
            print(f"  {size_kb:8.1f} KB  {f}")
else:
    print("No pdfs/ folder found!")

download_pdfs defined: True
safe_print defined: True

PDFs in folder: 46

CDF / Community / Project PDFs found:
      82.4 KB  2025-approved-Community-projects.pdf
     767.7 KB  2025-Community-projects-applications.pdf
     488.3 KB  List-of-CDF-approved-projects-for-2024.pdf
    5236.9 KB  Minutes-of-Community-engagement-and-budget-submission-2024.pdf
    3434.7 KB  Minutes-of-the-Community-Engagement-Meeting-and-Budget-Submission-2026.pdf
     103.4 KB  NOTICE-WARD-DEVELOPMENT-FUND.pdf


In [17]:
import pdfplumber

CDF_2024 = "pdfs/List-of-CDF-approved-projects-for-2024.pdf"

with pdfplumber.open(CDF_2024) as pdf:
    print(f"File: {CDF_2024}")
    print(f"Pages: {len(pdf.pages)}")
    
    # Page 1 text (truncated)
    print("\n--- PAGE 1 TEXT (first 1000 chars) ---")
    safe_print(pdf.pages[0].extract_text() or "(no text)", 1000)
    
    # Tables on page 1
    print("\n--- TABLES ON PAGE 1 ---")
    tables = pdf.pages[0].extract_tables()
    print(f"Found {len(tables)} table(s)")
    
    if tables:
        for i, row in enumerate(tables[0][:5]):
            print(f"Row {i}: {row}")

File: pdfs/List-of-CDF-approved-projects-for-2024.pdf
Pages: 1

--- PAGE 1 TEXT (first 1000 chars) ---
(no text)

--- TABLES ON PAGE 1 ---
Found 0 table(s)


In [18]:
import pdfplumber

CDF_2024 = "pdfs/List-of-CDF-approved-projects-for-2024.pdf"

with pdfplumber.open(CDF_2024) as pdf:
    page = pdf.pages[0]
    print("Page size:", page.width, "x", page.height)
    print("Images on page:", len(page.images))
    print("Chars extracted:", len(page.chars))
    print("Lines extracted:", len(page.lines))
    print("Rects extracted:", len(page.rects))
    
    if page.images:
        for img in page.images:
            print(f"  Image: {img['width']}x{img['height']} px")

Page size: 595.2 x 841.68
Images on page: 1
Chars extracted: 0
Lines extracted: 0
Rects extracted: 0
  Image: 587.52x829.44 px


In [19]:
!pip install pymupdf pillow pytesseract

   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/19.8 MB 192.7 kB/s eta 0:01:41
   - -------------------------------------- 0.5/19.8 MB 192.7 kB/s eta 0:01:41
  

In [21]:
import pymupdf # PyMuPDF
import os

os.makedirs("pdf_images", exist_ok=True)

doc = fitz.open(CDF_2024)
page = doc[0]
pix = page.get_pixmap(dpi=200)
img_path = "pdf_images/cdf_projects_page1.png"
pix.save(img_path)
print(f"Saved: {img_path}")
print(f"Image size: {pix.width} x {pix.height}")

Saved: pdf_images/cdf_projects_page1.png
Image size: 1654 x 2338


## Finding: CDF Projects PDF is a Scanned Image

**File:** `List-of-CDF-approved-projects-for-2024.pdf`  
**Diagnostic results:**
- Page size: 595 × 842 pts (A4)
- Embedded images: 1 (587 × 829 px)
- Character objects: 0
- Line objects: 0
- Table objects: 0
- Text extraction: empty

**Conclusion:** This is a raster image embedded in a PDF wrapper — 
not a digitally-typeset document. Standard pdfplumber extraction 
cannot work. Two paths forward:

1. **OCR** via `pytesseract` (Tesseract engine) — automated but may 
   lose table structure, especially with narrow columns.
2. **Manual transcription** of key rows — accurate but time-consuming.

**Decision:** Try OCR first. If accuracy is unacceptable, transcribe 
manually with the image open as reference.

In [23]:
import pdfplumber, os

print(f"{'File':65} {'Pages':>6} {'Text chars':>12}")
print("-" * 90)

for f in sorted(os.listdir("pdfs")):
    if not f.lower().endswith(".pdf"):
        continue
    path = f"pdfs/{f}"
    try:
        with pdfplumber.open(path) as pdf:
            total_chars = sum(len(p.extract_text() or "") for p in pdf.pages)
            print(f"{f[:65]:65} {len(pdf.pages):>6} {total_chars:>12,}")
    except Exception as e:
        print(f"{f[:65]:65} ERROR: {e}")

File                                                               Pages   Text chars
------------------------------------------------------------------------------------------
12-TH-ORDINARY-COUNCIL-28-JUNE-2024.pdf                               11       16,120
14TH-ORDINARY-COUNCIL-2024.pdf                                        64       84,322
2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf               4        2,295
2024-2026-OUTPUT-BASED-BUDGET-CHILILABOMBWE-HEAD-MANDATE-OBJECTIV     41       80,337
2024-Actionable-stakeholder-engagement-plan-1.pdf                      3            0
2025-CHILILABOMBWE-MUNICIPAL-COUNCIL-ADJUSTED-BUDGET.pdf              53      106,302
2025-Community-projects-applications.pdf                              10            0
2025-approved-Community-projects.pdf                                   2            0
Act-No.-3-The-Data-Protection-Act-2021_0-2.pdf                        40       69,170
Annual-Performance-Report-2024.pdf               

In [24]:
import pdfplumber

BUDGET_2026 = "pdfs/CHILILABOMBWE-MUNICIPAL-COUNCIL-2026-APPROVED-BUDGET.pdf"

with pdfplumber.open(BUDGET_2026) as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    print("\n--- FIRST 5 PAGES SUMMARY ---\n")
    
    for i in range(min(5, len(pdf.pages))):
        page = pdf.pages[i]
        text = page.extract_text() or ""
        tables = page.extract_tables()
        
        print(f"===== PAGE {i+1} =====")
        print(f"Text length: {len(text)} chars | Tables: {len(tables)}")
        
        # Show first 300 chars of text
        safe_print(text, 300)
        print()
        

Total pages: 213

--- FIRST 5 PAGES SUMMARY ---

===== PAGE 1 =====
Text length: 0 chars | Tables: 0


===== PAGE 2 =====
Text length: 0 chars | Tables: 0


===== PAGE 3 =====
Text length: 0 chars | Tables: 0


===== PAGE 4 =====
Text length: 0 chars | Tables: 0


===== PAGE 5 =====
Text length: 0 chars | Tables: 0




In [25]:
OBB = "pdfs/2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf"

with pdfplumber.open(OBB) as pdf:
    print(f"Pages: {len(pdf.pages)}\n")
    for i, page in enumerate(pdf.pages):
        print(f"===== PAGE {i+1} =====")
        safe_print(page.extract_text() or "(no text)", 1500)
        print(f"\nTables on this page: {len(page.extract_tables())}\n")

Pages: 4

===== PAGE 1 =====
CHILILABOMBWE MUNICIPAL COUNCIL
2024 OUTPUT BASED BUDGET
2024 PROGRAMMES/ SUB PROGRAMMES ALLOCATION
S/N PROGRAMMES/ SUB PROGRAMMES 2024 BUDGET
1 CONSTITUENCY DEVELOPMENT FUND 30,635,642
Community Projects 17,462,316
Youth & Women Empowerment 5,820,772
Secondary school boarding & Skill 5,820,772
Administrative Cost 1,531,782
2 LOCAL GOVERNANCE 10,157,466
Local elections 5,062,680
Legslative Function 3,525,996
Citizen Engagement 1,568,789
3 INTEGRATED DEVELOPMENT PLANNING 9,142,856
Social economic planning 6,280,021
Spatial Planning 2,646,189
Environmental Planning 216,646
4 ECONOMIC AND BUSINESS DEVELOPMENT 2,826,020
Trade Facilitation and Licencing 2,826,020
5 PUBLIC HEALTH AND ENVIRONMENTAL PROTECTION 18,926,013
Health Inspection 5,616,001
Solid waste Management 10,573,477
Pest control 319,211
Pollution control 79,765
Water supply and sanitation services 1,003,916
Stormy Water 222,150
Cemetery and funeral services 1,111,494
6 HOUSING AND COMMUNITY AMENITIE

In [26]:
import pdfplumber

OBB = "pdfs/2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf"

with pdfplumber.open(OBB) as pdf:
    # Page 1 has the table
    tables = pdf.pages[0].extract_tables()
    print(f"Tables found on page 1: {len(tables)}")
    
    if tables:
        for row in tables[0]:
            print(row)

Tables found on page 1: 1
['S/N', 'PROGRAMMES/ SUB PROGRAMMES', '2024 BUDGET']
['1 CONSTITUENCY DEVELOPMENT FUND 30,635,642', None, None]
['Community Projects 17,462,316\nYouth & Women Empowerment 5,820,772\nSecondary school boarding & Skill 5,820,772\nAdministrative Cost 1,531,782', None, None]
['2 LOCAL GOVERNANCE 10,157,466', None, None]
['Local elections 5,062,680\nLegslative Function 3,525,996\nCitizen Engagement 1,568,789', None, None]
['3 INTEGRATED DEVELOPMENT PLANNING 9,142,856', None, None]
['Social economic planning 6,280,021\nSpatial Planning 2,646,189\nEnvironmental Planning 216,646', None, None]
['4 ECONOMIC AND BUSINESS DEVELOPMENT 2,826,020', None, None]
['Trade Facilitation and Licencing 2,826,020', None, None]
['5 PUBLIC HEALTH AND ENVIRONMENTAL PROTECTION 18,926,013', None, None]
['Health Inspection 5,616,001\nSolid waste Management 10,573,477\nPest control 319,211\nPollution control 79,765\nWater supply and sanitation services 1,003,916\nStormy Water 222,150\nCemete

In [27]:
BUDGET_2025 = "pdfs/2025-CHILILABOMBWE-MUNICIPAL-COUNCIL-ADJUSTED-BUDGET.pdf"

with pdfplumber.open(BUDGET_2025) as pdf:
    print(f"Total pages: {len(pdf.pages)}\n")
    
    # Find pages with the most text (likely summary/budget tables)
    page_stats = []
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        tables = page.extract_tables()
        page_stats.append((i+1, len(text), len(tables)))
    
    # Show top 10 pages by text length
    print("Top 10 content-rich pages:")
    print(f"{'Page':>5} {'Text chars':>12} {'Tables':>8}")
    for pg, txt, tbl in sorted(page_stats, key=lambda x: -x[1])[:10]:
        print(f"{pg:>5} {txt:>12,} {tbl:>8}")

Total pages: 53

Top 10 content-rich pages:
 Page   Text chars   Tables
   10        5,055        0
    7        3,182        1
    1        2,903        0
   11        2,896        0
   18        2,861        2
   34        2,699        1
   14        2,588        1
   16        2,558        2
    9        2,523        1
   43        2,440        3


## Findings: Which PDFs Contain Extractable Data (2026-09-11)

A diagnostic scan of all 46 downloaded PDFs revealed a critical split:

### ✅ Text-Based PDFs (extractable with pdfplumber)
- `2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf` ← **PRIMARY**
- `2025-CHILILABOMBWE-MUNICIPAL-COUNCIL-ADJUSTED-BUDGET.pdf` ← **PRIMARY**
- `2024-2026-OUTPUT-BASED-BUDGET-...-SUB-PROGRAMMES.pdf`
- `CHILILABOMBWE-DISTRICT-IDP-2022-2031.pdf`
- Council minutes (11th, 12th, 13th, 14th, 1st, 2nd)
- Service Charter
- Legal acts (CDF Act, Public Finance Act, etc.)

### ❌ Scanned/Image PDFs (need OCR or manual work)
- `List-of-CDF-approved-projects-for-2024.pdf` (1 page — manual)
- `2025-approved-Community-projects.pdf` (2 pages — manual)
- `CHILILABOMBWE-MUNICIPAL-COUNCIL-2026-APPROVED-BUDGET.pdf` (213 pages — too large to OCR)
- 7 × Financial Statements (2018–2024)
- Annual / Bi-Annual Performance Reports
- Multiple stakeholder meeting minutes

### Decision
Focus dataset creation on text-based PDFs. Supplement with manual 
transcription of the 1-page CDF projects list.

In [28]:
import pandas as pd
import re
import pdfplumber

# --- Re-open and extract the table ---
OBB = "pdfs/2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf"

with pdfplumber.open(OBB) as pdf:
    raw_table = pdf.pages[0].extract_tables()[0]

# --- Parse into rows ---
rows = []
current_programme = None

def parse_amount(text):
    """Extract a number from a string like 'Community Projects 17,462,316'."""
    m = re.search(r'([\d,]+)\s*$', text.strip())
    if m:
        return int(m.group(1).replace(",", ""))
    return None

def split_name_amount(text):
    """Split 'Name 123,456' into ('Name', 123456)."""
    m = re.match(r'^(.*?)\s+([\d,]+)\s*$', text.strip())
    if m:
        return m.group(1).strip(), int(m.group(2).replace(",", ""))
    return text.strip(), None

# Skip header row
for row in raw_table[1:]:
    cell = row[0]
    if not cell:
        continue
    
    lines = cell.split("\n")
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        # A parent programme starts with a number: "1 CONSTITUENCY..."
        m = re.match(r'^(\d+)\s+(.+)', line)
        if m:
            # This is a parent programme
            current_programme = m.group(2)
            name, amount = split_name_amount(current_programme)
            rows.append({
                "level": 1,
                "programme_code": m.group(1),
                "programme": name,
                "sub_programme": None,
                "amount_zmw": amount,
            })
        else:
            # This is a sub-programme
            name, amount = split_name_amount(line)
            rows.append({
                "level": 2,
                "programme_code": None,
                "programme": current_programme.split(" ", 1)[-1] if current_programme else None,
                "sub_programme": name,
                "amount_zmw": amount,
            })

df = pd.DataFrame(rows)
print(f"Total rows parsed: {len(df)}")
print(f"  - Parent programmes: {(df['level'] == 1).sum()}")
print(f"  - Sub-programmes:    {(df['level'] == 2).sum()}")
print()
print(df.head(20))

Total rows parsed: 45
  - Parent programmes: 10
  - Sub-programmes:    35

    level programme_code                                       programme  \
0       1              1                   CONSTITUENCY DEVELOPMENT FUND   
1       2            NaN                     DEVELOPMENT FUND 30,635,642   
2       2            NaN                     DEVELOPMENT FUND 30,635,642   
3       2            NaN                     DEVELOPMENT FUND 30,635,642   
4       2            NaN                     DEVELOPMENT FUND 30,635,642   
5       1              2                                LOCAL GOVERNANCE   
6       2            NaN                           GOVERNANCE 10,157,466   
7       2            NaN                           GOVERNANCE 10,157,466   
8       2            NaN                           GOVERNANCE 10,157,466   
9       1              3                 INTEGRATED DEVELOPMENT PLANNING   
10      2            NaN                  DEVELOPMENT PLANNING 9,142,856   
11      2    

In [29]:
df.to_csv(
    "db-unza26-csc4792-chililabombwe_obb_budget_2024.csv",
    sep="|",
    index=False
)
print("Saved: db-unza26-csc4792-chililabombwe_obb_budget_2024.csv")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")

Saved: db-unza26-csc4792-chililabombwe_obb_budget_2024.csv
Rows: 45, Columns: 5


In [30]:
# Read it back to confirm
check = pd.read_csv("db-unza26-csc4792-chililabombwe_obb_budget_2024.csv", sep="|")
print(f"Loaded {len(check)} rows")
print(check.head())

Loaded 45 rows
   level  programme_code                      programme  \
0      1             1.0  CONSTITUENCY DEVELOPMENT FUND   
1      2             NaN    DEVELOPMENT FUND 30,635,642   
2      2             NaN    DEVELOPMENT FUND 30,635,642   
3      2             NaN    DEVELOPMENT FUND 30,635,642   
4      2             NaN    DEVELOPMENT FUND 30,635,642   

                       sub_programme  amount_zmw  
0                                NaN    30635642  
1                 Community Projects    17462316  
2          Youth & Women Empowerment     5820772  
3  Secondary school boarding & Skill     5820772  
4                Administrative Cost     1531782  


In [31]:
budget_comparison = pd.DataFrame([
    {"metric": "2024_budget_zmw",          "value": 147958491},
    {"metric": "2023_budget_zmw",          "value": 127863665},
    {"metric": "budget_increment_zmw",     "value": 20094826},
    {"metric": "increase_percentage",      "value": 16},
])

budget_comparison.to_csv(
    "db-unza26-csc4792-chililabombwe_budget_comparison_2023_2024.csv",
    sep="|",
    index=False
)
print("Saved comparison CSV")
print(budget_comparison)

Saved comparison CSV
                 metric      value
0       2024_budget_zmw  147958491
1       2023_budget_zmw  127863665
2  budget_increment_zmw   20094826
3   increase_percentage         16


## First Extractions Complete — 2026-09-11

### CSV 1: `db-unza26-csc4792-chililabombwe_obb_budget_2024.csv`
- **Source:** `2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf`
- **Method:** pdfplumber `extract_tables()` + regex parsing
- **Rows:** [fill in]
- **Columns:** level, programme_code, programme, sub_programme, amount_zmw
- **Coverage:** 18 parent programmes + sub-programmes = full 2024 council budget

### CSV 2: `db-unza26-csc4792-chililabombwe_budget_comparison_2023_2024.csv`
- **Source:** Page 2 of same PDF
- **Rows:** 4
- **Columns:** metric, value
- **Coverage:** Year-over-year budget growth

### Next Steps
1. Extract summary tables from 2025 Adjusted Budget (pages 7, 9, 10, 11, 14, 16, 18)
2. Manually transcribe the 1-page CDF projects PDF
3. Build combined dataset for Kaggle

In [32]:
BUDGET_2025 = "pdfs/2025-CHILILABOMBWE-MUNICIPAL-COUNCIL-ADJUSTED-BUDGET.pdf"

with pdfplumber.open(BUDGET_2025) as pdf:
    for page_num in [7, 9, 10, 11, 14, 16, 18, 34, 43]:
        page = pdf.pages[page_num - 1]
        print(f"\n===== PAGE {page_num} =====")
        safe_print(page.extract_text() or "(no text)", 800)


===== PAGE 7 =====
OUTPUT BASED ANNUAL BUDGET Page 7
HEAD9101 CHILILABOMBWE MUNICIPAL COUNCIL
Table 3: Budget Allocation by Programme and Sub-Programme
PROGRAMME/SUB-PROGRAMME 2023 BUDGET 2024 BUDGET 2025 BUDGET
Approved Expendit Approved Expendit Estimate
ure ure*
(0) (0)
1 Constituency Development (0) 30,635,642 36,058,150
779 Community Projects - (1) (0) (0) 17,462,316 (0) 21,962,691
780 Women and Youth Empowerment - (3) (0) (0) 5,820,772 (0) 6,228,226
781 CDF Administration - (5) (0) (0) 1,531,782 (0) 1,639,007
782 Secondary School and Skills Development Bursaries - (7) (0) (0) 5,820,772 (0) 6,228,226
(0) (0)
2 Local Governance (0) 10,157,466 12,077,969
040 (cid:9)Local elections (0) (0) 5,062,680 (0) 6,962,502
044 Legislative Function - (9) (0) (0) 3,525,996 (0) 3,680,291
045 Citizen Engagement (0) (0) 
... [2382 more characters truncated]

===== PAGE 9 =====
OUTPUT BASED ANNUAL BUDGET Page 9
HEAD9101 CHILILABOMBWE MUNICIPAL COUNCIL
Head Total (0) (0) 143,066,945 (0) 223,603,854


In [33]:
import pandas as pd
import re
import pdfplumber

# --- Re-open and extract ---
OBB = "pdfs/2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf"
with pdfplumber.open(OBB) as pdf:
    raw_table = pdf.pages[0].extract_tables()[0]

def split_name_amount(text):
    """Split 'Name 123,456' into ('Name', 123456)."""
    text = text.strip()
    m = re.match(r'^(.*?)\s+([\d,]+)\s*$', text)
    if m:
        return m.group(1).strip(), int(m.group(2).replace(",", ""))
    return text, None

rows = []
current_code = None
current_name = None

for row in raw_table[1:]:          # skip header
    cell = row[0]
    if not cell:
        continue
    for line in cell.split("\n"):
        line = line.strip()
        if not line:
            continue
        
        # Parent programme? starts with a number
        m = re.match(r'^(\d+)\s+(.+)', line)
        if m:
            current_code = m.group(1)
            name, amount = split_name_amount(m.group(2))
            current_name = name
            rows.append({
                "level": 1,
                "programme_code": current_code,
                "programme": name,
                "sub_programme": None,
                "amount_zmw": amount,
            })
        else:
            # Sub-programme
            name, amount = split_name_amount(line)
            rows.append({
                "level": 2,
                "programme_code": current_code,   # inherit parent's code
                "programme": current_name,        # ✅ full parent name (not split)
                "sub_programme": name,
                "amount_zmw": amount,
            })

df = pd.DataFrame(rows)
print(f"Total rows: {len(df)}")
print(f"  Parent programmes: {(df['level'] == 1).sum()}")
print(f"  Sub-programmes:    {(df['level'] == 2).sum()}")
print(f"  Total budget sum (parent rows only): {df[df['level']==1]['amount_zmw'].sum():,} ZMW")
print()
print(df.head(15).to_string())

Total rows: 45
  Parent programmes: 10
  Sub-programmes:    35
  Total budget sum (parent rows only): 135,762,103 ZMW

    level programme_code                          programme                      sub_programme  amount_zmw
0       1              1      CONSTITUENCY DEVELOPMENT FUND                                NaN    30635642
1       2              1      CONSTITUENCY DEVELOPMENT FUND                 Community Projects    17462316
2       2              1      CONSTITUENCY DEVELOPMENT FUND          Youth & Women Empowerment     5820772
3       2              1      CONSTITUENCY DEVELOPMENT FUND  Secondary school boarding & Skill     5820772
4       2              1      CONSTITUENCY DEVELOPMENT FUND                Administrative Cost     1531782
5       1              2                   LOCAL GOVERNANCE                                NaN    10157466
6       2              2                   LOCAL GOVERNANCE                    Local elections     5062680
7       2              2 

In [35]:
import pandas as pd
import re
import pdfplumber

OBB = "pdfs/2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf"

def split_name_amount(text):
    text = text.strip()
    m = re.match(r'^(.*?)\s+([\d,]+)\s*$', text)
    if m:
        return m.group(1).strip(), int(m.group(2).replace(",", ""))
    return text, None

rows = []
current_code, current_name = None, None

with pdfplumber.open(OBB) as pdf:
    for page_idx, page in enumerate(pdf.pages[:2]):   # pages 1 AND 2
        print(f"\n--- Processing page {page_idx + 1} ---")
        tables = page.extract_tables()
        if not tables:
            print("  No tables found")
            continue
        for row in tables[0][1:]:                    # skip header
            cell = row[0]
            if not cell:
                continue
            for line in cell.split("\n"):
                line = line.strip()
                if not line:
                    continue
                # Stop at the comparison section header
                if "BUDGET COMPARISON" in line.upper():
                    break
                # Skip totals
                if line.upper().startswith("TOTAL"):
                    continue
                m = re.match(r'^(\d+)\s+(.+)', line)
                if m:
                    current_code = m.group(1)
                    name, amount = split_name_amount(m.group(2))
                    current_name = name
                    rows.append({
                        "level": 1,
                        "programme_code": current_code,
                        "programme": name,
                        "sub_programme": None,
                        "amount_zmw": amount,
                    })
                else:
                    name, amount = split_name_amount(line)
                    rows.append({
                        "level": 2,
                        "programme_code": current_code,
                        "programme": current_name,
                        "sub_programme": name,
                        "amount_zmw": amount,
                    })

df = pd.DataFrame(rows)
print(f"\nTotal rows: {len(df)}")
print(f"Parent programmes: {(df['level'] == 1).sum()}")
print(f"Sub-programmes:    {(df['level'] == 2).sum()}")
parent_total = df[df['level'] == 1]['amount_zmw'].sum()
print(f"\nParent total: {parent_total:,} ZMW  (expected 147,958,491)")
print(f"Difference:   {parent_total - 147958491:+,} ZMW")
print("\nAll parent programmes:")
print(df[df['level'] == 1][['programme_code', 'programme', 'amount_zmw']].to_string(index=False))


--- Processing page 1 ---

--- Processing page 2 ---
  No tables found

Total rows: 45
Parent programmes: 10
Sub-programmes:    35

Parent total: 135,762,103 ZMW  (expected 147,958,491)
Difference:   -12,196,388 ZMW

All parent programmes:
programme_code                                  programme  amount_zmw
             1              CONSTITUENCY DEVELOPMENT FUND    30635642
             2                           LOCAL GOVERNANCE    10157466
             3            INTEGRATED DEVELOPMENT PLANNING     9142856
             4          ECONOMIC AND BUSINESS DEVELOPMENT     2826020
             5 PUBLIC HEALTH AND ENVIRONMENTAL PROTECTION    18926013
             6            HOUSING AND COMMUNITY AMENITIES    16833566
             7            RECREATION CULTURE AND RELIGION     2284261
             8            EDUCATION AND SKILL DEVELOPMENT      461596
            10                    PUBLIC ORDER AND SAFETY    10784546
            11            MANAGEMENT AND SUPPORT SERVICES  

In [1]:
import pdfplumber

OBB = "pdfs/2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf"

with pdfplumber.open(OBB) as pdf:
    for i in range(len(pdf.pages)):
        page = pdf.pages[i]
        tables = page.extract_tables()
        text = page.extract_text() or ""
        print(f"--- Page {i+1} ---")
        print(f"  Text chars: {len(text)}")
        print(f"  Tables: {len(tables)}")
        if tables:
            print(f"  Table 1: {len(tables[0])} rows × {len(tables[0][0])} cols")
            print(f"  First 3 rows:")
            for row in tables[0][:3]:
                print(f"    {row}")
        print()

--- Page 1 ---
  Text chars: 1651
  Tables: 1
  Table 1: 21 rows × 3 cols
  First 3 rows:
    ['S/N', 'PROGRAMMES/ SUB PROGRAMMES', '2024 BUDGET']
    ['1 CONSTITUENCY DEVELOPMENT FUND 30,635,642', None, None]
    ['Community Projects 17,462,316\nYouth & Women Empowerment 5,820,772\nSecondary school boarding & Skill 5,820,772\nAdministrative Cost 1,531,782', None, None]

--- Page 2 ---
  Text chars: 644
  Tables: 0

--- Page 3 ---
  Text chars: 0
  Tables: 0

--- Page 4 ---
  Text chars: 0
  Tables: 0



In [2]:
import re
import pdfplumber
import pandas as pd

OBB = "pdfs/2024-2026-CHILILABOMBWE-MUNICIAL-COUNCIL-OBB-SUMMARY.pdf"

def split_name_amount(text):
    text = text.strip()
    m = re.match(r'^(.*?)\s+([\d,]+)\s*$', text)
    if m:
        return m.group(1).strip(), int(m.group(2).replace(",", ""))
    return text, None

rows = []
current_code, current_name = None, None

def process_line(line):
    """Process one line of text — either a parent or sub-programme row."""
    global current_code, current_name
    line = line.strip()
    if not line:
        return
    
    # Skip junk
    if "BUDGET COMPARISON" in line.upper():
        return
    if line.upper().startswith("CHILILABOMBWE MUNICIPAL"):
        return
    if line.upper().startswith("TOTAL"):
        return
    if re.match(r'^20\d\d\s+(AND|BUDGET|BUDGETED|BUDGET\s+AMOUNT|INCREMENT)', line.upper()):
        return
    if "INCREASE IN PERCENTAGE" in line.upper():
        return
    
    # Parent programme? Starts with a number followed by text
    m = re.match(r'^(\d+)\s+([A-Z][A-Z\s,&]+?)\s+([\d,]+)\s*$', line)
    if m:
        current_code = m.group(1)
        current_name = m.group(2).strip()
        rows.append({
            "level": 1,
            "programme_code": current_code,
            "programme": current_name,
            "sub_programme": None,
            "amount_zmw": int(m.group(3).replace(",", "")),
        })
        return
    
    # Sub-programme? "Name 123,456"
    name, amount = split_name_amount(line)
    if amount is not None and current_code is not None:
        rows.append({
            "level": 2,
            "programme_code": current_code,
            "programme": current_name,
            "sub_programme": name,
            "amount_zmw": amount,
        })

with pdfplumber.open(OBB) as pdf:
    # --- Page 1: extract from table ---
    page1 = pdf.pages[0]
    tables = page1.extract_tables()
    if tables:
        for row in tables[0][1:]:          # skip header
            cell = row[0]
            if cell:
                for line in cell.split("\n"):
                    process_line(line)
    
    # --- Page 2: extract from raw text ---
    page2_text = pdf.pages[1].extract_text() or ""
    for line in page2_text.split("\n"):
        process_line(line)

df = pd.DataFrame(rows)
parent_total = df[df['level'] == 1]['amount_zmw'].sum()

print(f"Total rows: {len(df)}")
print(f"Parent programmes: {(df['level'] == 1).sum()}")
print(f"Sub-programmes:    {(df['level'] == 2).sum()}")
print(f"\nParent total: {parent_total:,} ZMW  (expected 147,958,491)")
print(f"Difference:   {parent_total - 147958491:+,} ZMW")
print()
print(df[df['level'] == 1][['programme_code', 'programme', 'amount_zmw']].to_string(index=False))

Total rows: 60
Parent programmes: 17
Sub-programmes:    43

Parent total: 147,958,491 ZMW  (expected 147,958,491)
Difference:   +0 ZMW

programme_code                                  programme  amount_zmw
             1              CONSTITUENCY DEVELOPMENT FUND    30635642
             2                           LOCAL GOVERNANCE    10157466
             3            INTEGRATED DEVELOPMENT PLANNING     9142856
             4          ECONOMIC AND BUSINESS DEVELOPMENT     2826020
             5 PUBLIC HEALTH AND ENVIRONMENTAL PROTECTION    18926013
             6            HOUSING AND COMMUNITY AMENITIES    16833566
             7            RECREATION CULTURE AND RELIGION     2284261
             8            EDUCATION AND SKILL DEVELOPMENT      461596
            10                    PUBLIC ORDER AND SAFETY    10784546
            11            MANAGEMENT AND SUPPORT SERVICES    33710137
            12       RESOURCE MOBILISATION AND MANAGEMENT     6545962
            13          

In [3]:
df.to_csv(
    "db-unza26-csc4792-chililabombwe_obb_budget_2024.csv",
    sep="|",
    index=False
)
print("✅ Saved: db-unza26-csc4792-chililabombwe_obb_budget_2024.csv")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
print(f"File size: {os.path.getsize('db-unza26-csc4792-chililabombwe_obb_budget_2024.csv')} bytes")

✅ Saved: db-unza26-csc4792-chililabombwe_obb_budget_2024.csv
Rows: 60, Columns: 5


NameError: name 'os' is not defined

In [5]:
import os
import pandas as pd

# Verify the file exists
fname = "db-unza26-csc4792-chililabombwe_obb_budget_2024.csv"
print(f"File exists: {os.path.exists(fname)}")
print(f"Size: {os.path.getsize(fname)} bytes")

File exists: True
Size: 3518 bytes


In [6]:
import pandas as pd

comparison = pd.DataFrame([
    {"metric": "2024_budget_zmw",       "value": 147958491},
    {"metric": "2023_budget_zmw",       "value": 127863665},
    {"metric": "budget_increment_zmw",  "value": 20094826},
    {"metric": "increase_percentage",   "value": 16},
])
comparison.to_csv(
    "db-unza26-csc4792-chililabombwe_budget_comparison_2023_2024.csv",
    sep="|",
    index=False
)
print("✅ Saved comparison CSV")
print(comparison.to_string(index=False))

✅ Saved comparison CSV
              metric     value
     2024_budget_zmw 147958491
     2023_budget_zmw 127863665
budget_increment_zmw  20094826
 increase_percentage        16


In [7]:
check = pd.read_csv("db-unza26-csc4792-chililabombwe_obb_budget_2024.csv", sep="|")
print(f"Rows loaded back: {len(check)}")
print(f"Total from CSV (parents only): {check[check['level']==1]['amount_zmw'].sum():,} ZMW")
print("\nFirst 10 rows:")
print(check.head(10).to_string(index=False))

Rows loaded back: 60
Total from CSV (parents only): 147,958,491 ZMW

First 10 rows:
 level  programme_code                       programme                     sub_programme  amount_zmw
     1               1   CONSTITUENCY DEVELOPMENT FUND                               NaN    30635642
     2               1   CONSTITUENCY DEVELOPMENT FUND                Community Projects    17462316
     2               1   CONSTITUENCY DEVELOPMENT FUND         Youth & Women Empowerment     5820772
     2               1   CONSTITUENCY DEVELOPMENT FUND Secondary school boarding & Skill     5820772
     2               1   CONSTITUENCY DEVELOPMENT FUND               Administrative Cost     1531782
     1               2                LOCAL GOVERNANCE                               NaN    10157466
     2               2                LOCAL GOVERNANCE                   Local elections     5062680
     2               2                LOCAL GOVERNANCE               Legslative Function     3525996
     2 

In [9]:
import pandas as pd

cdf_data = [
    # item_no, category, project_name, ward, allocation_zmw
    ("1",    "CAPITAL PROJECTS", "CAPITAL PROJECTS (parent)",                                  None,                   None),
    ("1.1",  "CAPITAL PROJECTS", "Procurement of 600 school desks",                             None,                   1200000.00),
    ("1.2",  "CAPITAL PROJECTS", "Construction of market shelter in Nakatindi ward",            "Nakatindi",            2504486.60),
    ("1.3",  "CAPITAL PROJECTS", "Construction of market shelter in Kamenza East ward",         "Kamenza East",         2549395.20),
    ("1.4",  "CAPITAL PROJECTS", "Construction of market shelter in the Central Business District", "CBD",              1381190.25),
    ("1.5",  "CAPITAL PROJECTS", "Construction of boundary fence around James Phiri Clinic",    None,                   182541.70),
    ("1.6",  "CAPITAL PROJECTS", "Construction of Health Post at Butondo",                      "Butondo",              1194996.81),
    ("1.7",  "CAPITAL PROJECTS", "Rural Electrification",                                       None,                   1000000.00),
    ("1.8",  "CAPITAL PROJECTS", "Procurement of Ambulance",                                    None,                   2300000.00),
    ("1.9",  "CAPITAL PROJECTS", "Procurement of YVTC Bricklaying equipment",                   None,                   530000.00),
    ("1.10", "CAPITAL PROJECTS", "Road maintenance in selected wards",                          None,                   1523236.88),
    ("1.11", "CAPITAL PROJECTS", "Procurement of TLB",                                          None,                   1800000.00),
    ("1.12", "CAPITAL PROJECTS", "Completion of Kamenza Police Post",                           "Kamenza",              423352.68),
]

df_cdf = pd.DataFrame(cdf_data, columns=[
    "item_no", "category", "project_name", "ward", "allocation_zmw"
])

# Add derived columns
df_cdf["sector"] = df_cdf["project_name"].apply(lambda x: (
    "education"  if any(k in x.lower() for k in ["school", "desk", "yvct", "yvct"]) else
    "markets"    if "market" in x.lower() else
    "health"     if any(k in x.lower() for k in ["clinic", "health", "ambulance", "hospital"]) else
    "water"      if any(k in x.lower() for k in ["water", "sanitation", "borehole"]) else
    "roads"      if any(k in x.lower() for k in ["road", "drainage"]) else
    "energy"     if any(k in x.lower() for k in ["electric", "power", "solar"]) else
    "safety"     if any(k in x.lower() for k in ["police", "fire"]) else
    "equipment"  if any(k in x.lower() for k in ["procurement", "equipment", "tlb", "vehicle"]) else
    "infrastructure"
))
df_cdf["status"] = "approved"
df_cdf["year"]   = 2024
df_cdf["fund"]   = "CDF"

# Save
df_cdf.to_csv(
    "db-unza26-csc4792-chililabombwe_cdf_projects_2024.csv",
    sep="|",
    index=False
)

# Summary
projects_only = df_cdf[df_cdf["allocation_zmw"].notna()]
print(f"✅ Saved CDF projects CSV")
print(f"Total rows: {len(df_cdf)}")
print(f"Projects (with allocations): {len(projects_only)}")
print(f"Total CDF capital allocation: K{projects_only['allocation_zmw'].sum():,.2f}")
print(f"\nBy sector:")
print(projects_only.groupby("sector")["allocation_zmw"].agg(["count", "sum"]).to_string())
print()
print(df_cdf.to_string(index=False))

✅ Saved CDF projects CSV
Total rows: 13
Projects (with allocations): 12
Total CDF capital allocation: K16,589,200.12

By sector:
                count         sum
sector                           
education           1  1200000.00
equipment           2  2330000.00
health              3  3677538.51
infrastructure      1  1000000.00
markets             3  6435072.05
roads               1  1523236.88
safety              1   423352.68

item_no         category                                                    project_name         ward  allocation_zmw         sector   status  year fund
      1 CAPITAL PROJECTS                                       CAPITAL PROJECTS (parent)          NaN             NaN infrastructure approved  2024  CDF
    1.1 CAPITAL PROJECTS                                 Procurement of 600 school desks          NaN      1200000.00      education approved  2024  CDF
    1.2 CAPITAL PROJECTS                Construction of market shelter in Nakatindi ward    Nakatindi    

## ✅ CSV #3: 2024 CDF Projects — Manual Transcription

**Source:** `List-of-CDF-approved-projects-for-2024.pdf` (scanned image)
**Output CSV:** `db-unza26-csc4792-chililabombwe_cdf_projects_2024.csv`

### Why manual transcription?
The source PDF contained a single embedded image (587×829 px) with zero 
extractable text characters. Standard text-based PDF extraction failed.

### Method
1. Converted PDF page to 300 DPI PNG using PyMuPDF.
2. Read the image directly and transcribed 12 project rows.
3. Verified each allocation value against the source image.
4. Added derived columns:
   - `sector` — classified by keyword matching on project name
   - `status` — set to "approved"
   - `year` — 2024
   - `fund` — "CDF"

### Data quality
- 100% accuracy for item numbers and allocations (visually verified)
- Ward names captured where present in source; blank otherwise
- Corrected one item number in the source (1.1 → 1.10)

### Reuse value
Enables analysis of:
- CDF project distribution by sector
- Geographic concentration (by ward)
- CDF allocation patterns at the local level
- Comparison with other Zambian councils' CDF projects

In [10]:
import pdfplumber

BUDGET_2025 = "pdfs/2025-CHILILABOMBWE-MUNICIPAL-COUNCIL-ADJUSTED-BUDGET.pdf"

with pdfplumber.open(BUDGET_2025) as pdf:
    print(f"Total pages: {len(pdf.pages)}\n")
    
    for page_num in [1, 7, 9, 10, 11, 14, 16, 18, 34, 43]:
        if page_num > len(pdf.pages):
            continue
        page = pdf.pages[page_num - 1]
        text = page.extract_text() or ""
        tables = page.extract_tables()
        print(f"===== PAGE {page_num} — {len(text)} chars, {len(tables)} tables =====")
        safe_print(text, 500)
        print()

Total pages: 53

===== PAGE 1 — 2903 chars, 0 tables =====
OUTPUT BASED ANNUAL BUDGET Page 1
HEAD9101 CHILILABOMBWE MUNICIPAL COUNCIL
1.0 MANDATE
(cid:9)To provide municipal services through operational and service excellence, innovation, community
engagement and observance of good financial management and accountability. This is in agreement
with the Republican Constitution (Amendment) Act No.2 of 2016 Part IX on the System of Devolved
Governance [Article 147 (2)] and Part XI on the System of Local Government.
2.0 STRATEGY
Chililabombwe Municipal Counc
... [2403 more characters truncated]

===== PAGE 7 — 3182 chars, 1 tables =====
OUTPUT BASED ANNUAL BUDGET Page 7
HEAD9101 CHILILABOMBWE MUNICIPAL COUNCIL
Table 3: Budget Allocation by Programme and Sub-Programme
PROGRAMME/SUB-PROGRAMME 2023 BUDGET 2024 BUDGET 2025 BUDGET
Approved Expendit Approved Expendit Estimate
ure ure*
(0) (0)
1 Constituency Development (0) 30,635,642 36,058,150
779 Community Projects - (1) (0) (0) 17,462,316 (0) 

In [11]:
import pdfplumber

BUDGET_2025 = "pdfs/2025-CHILILABOMBWE-MUNICIPAL-COUNCIL-ADJUSTED-BUDGET.pdf"

with pdfplumber.open(BUDGET_2025) as pdf:
    page = pdf.pages[6]   # page 7 (0-indexed)
    
    print("--- RAW TEXT (first 3000 chars) ---")
    safe_print(page.extract_text(), 3000)
    
    print("\n\n--- TABLES ---")
    tables = page.extract_tables()
    print(f"Tables found: {len(tables)}")
    for i, table in enumerate(tables):
        print(f"\n=== TABLE {i+1} — {len(table)} rows × {len(table[0])} cols ===")
        for row in table:
            print(row)

--- RAW TEXT (first 3000 chars) ---
OUTPUT BASED ANNUAL BUDGET Page 7
HEAD9101 CHILILABOMBWE MUNICIPAL COUNCIL
Table 3: Budget Allocation by Programme and Sub-Programme
PROGRAMME/SUB-PROGRAMME 2023 BUDGET 2024 BUDGET 2025 BUDGET
Approved Expendit Approved Expendit Estimate
ure ure*
(0) (0)
1 Constituency Development (0) 30,635,642 36,058,150
779 Community Projects - (1) (0) (0) 17,462,316 (0) 21,962,691
780 Women and Youth Empowerment - (3) (0) (0) 5,820,772 (0) 6,228,226
781 CDF Administration - (5) (0) (0) 1,531,782 (0) 1,639,007
782 Secondary School and Skills Development Bursaries - (7) (0) (0) 5,820,772 (0) 6,228,226
(0) (0)
2 Local Governance (0) 10,157,466 12,077,969
040 (cid:9)Local elections (0) (0) 5,062,680 (0) 6,962,502
044 Legislative Function - (9) (0) (0) 3,525,996 (0) 3,680,291
045 Citizen Engagement (0) (0) 1,568,789 (0) 1,435,176
(0) (0)
3 Integrated Development Planning (0) 9,142,856 5,339,802
006 Environmental planning - (11) (0) (0) 216,646 (0) 229,685
021 Spatial 

In [12]:
with pdfplumber.open(BUDGET_2025) as pdf:
    page = pdf.pages[13]   # page 14
    print("--- PAGE 14 TEXT ---")
    safe_print(page.extract_text(), 2500)
    
    print("\n--- TABLES ---")
    tables = page.extract_tables()
    for i, table in enumerate(tables):
        print(f"\n=== TABLE {i+1} — {len(table)} rows × {len(table[0])} cols ===")
        for row in table:
            print(row)

--- PAGE 14 TEXT ---
Page 14 OUTPUT BASED ANNUAL BUDGET
HEAD9101 CHILILABOMBWE MUNICIPAL COUNCIL
Programme: 1Constituency Development
Table 6: Programme Outputs
Key Output and Output Indicator 2023 2024 2025
Target Actual Target Actual* Target
School Desks Procured
01 Number of school desks procured (0) - - (0) 600
Maternity Annexes Constructed
01 Number of maternity annexes constructed (0) (0) (0) (0) 2
Township Roads Rehabilitated
01 Kilometres of township roads rehabilitated (0) (0) (0) (0) 50
1*3 Classroom Block Completed
01 Number of 1*3 classroom blocks completed (0) (0) (0) (0) 3
Boreholes Drilled
01 Number of boreholes drilled (0) (0) (0) (0) 3
Youth,Women and Community Groups Empowered with Grants
01 Number of youth, women and community groups empowered with grants (0) (0) (0) (0) 160
Youth,Women and Community Groups Empowered with Loans
01 Number of youth,yomen and community groups empowered with loans (0) (0) (0) (0) 100
CDF Projects Appraised and Monitored.
01 Number of CDF

In [13]:
with pdfplumber.open(BUDGET_2025) as pdf:
    page = pdf.pages[8]   # page 9
    print("--- PAGE 9 TEXT ---")
    safe_print(page.extract_text(), 2500)

--- PAGE 9 TEXT ---
OUTPUT BASED ANNUAL BUDGET Page 9
HEAD9101 CHILILABOMBWE MUNICIPAL COUNCIL
Head Total (0) (0) 143,066,945 (0) 223,603,854
(1)
GRZ - CDF Grant 18,684,678
(3)
GRZ - CDF Grant 2,491,290
(5)
GRZ - CDF Grant 1,522,925
(7)
GRZ - CDF Grant 6,228,226
(9)
GRZ - LGEF Grant 1,317,940
(11)
GRZ - LGEF Grant 177,989
(13)
GRZ - LGEF Grant 585,413
(15)
GRZ - LGEF Grant 94,780
(17)
GRZ - LGEF Grant 3,949,368
(19)
GRZ - LGEF Grant 915,305
(21)
GRZ - LGEF Grant 1,439
(23)
GRZ - LGEF Grant 404,780
(25)
GRZ - LGEF Grant 11,350
(27)
GRZ - LGEF Grant 2,613
(29)
GRZ - LGEF Grant 4,948,815
(31)
GRZ - LGEF Local R 56,269
(33)
GRZ - LGEF Grant 1,109,854
(35)
GRZ - LGEF Grant 402,327
(37)
GRZ - LGEF Grant 313,432
(39)
GRZ - LGEF Grant 757,487
(41)
GRZ - LGEF Grant 12,570,771
(43)
GRZ Grant 9,850
(45)
GRZ - Other Grant 50,000
Grants
The budget allocation by Programme and Sub programme shows that Constituency Development Fund
Programme has a total budget of K36.1 million representing 16.13 perce

In [16]:
import re
import pdfplumber
import pandas as pd

BUDGET_2025 = "pdfs/2025-CHILILABOMBWE-MUNICIPAL-COUNCIL-ADJUSTED-BUDGET.pdf"

# --- Extract from pages 7 AND 8 ---
with pdfplumber.open(BUDGET_2025) as pdf:
    all_lines = []
    for pnum in [6, 7]:     # pages 7 and 8 (0-indexed 6, 7)
        text = pdf.pages[pnum].extract_text() or ""
        all_lines.extend(text.split("\n"))

rows = []
current_programme = None
current_code = None

def extract_amounts(s):
    found = re.findall(r'\b[\d,]{5,}\b', s)
    return [int(a.replace(",", "")) for a in found]

for raw_line in all_lines:
    line = raw_line.strip()
    if not line or "(cid:" in line:
        continue
    if "PROGRAMME" in line or "Table" in line or ("BUDGET" in line and "202" in line):
        continue
    if line.startswith("OUTPUT BASED") or line.startswith("HEAD"):
        continue
    
    # Parent programme
    m_par = re.match(r'^(\d{1,2})\s+([A-Z][A-Za-z\s,&]+?)\s+\(', line)
    if m_par:
        code = m_par.group(1)
        name = m_par.group(2).strip()
        amounts = extract_amounts(line)
        if amounts and amounts[-1] > 1_000_000:
            current_programme = name
            current_code = code
            rows.append({
                "level": 1, "code": code, "programme": name,
                "sub_programme": None,
                "budget_2023_zmw": amounts[-2] if len(amounts) >= 2 else None,
                "budget_2024_zmw": None,
                "budget_2025_zmw": amounts[-1],
            })
        continue
    
    # Sub-programme
    m_sub = re.match(r'^(\d{3})\s*([A-Z][A-Za-z\s,&\.]+?)\s+(?:-\s*)?\(?\d', line)
    if m_sub:
        code = m_sub.group(1)
        name = m_sub.group(2).strip().rstrip(",")
        amounts = extract_amounts(line)
        rows.append({
            "level": 2, "code": code, "programme": current_programme,
            "sub_programme": name,
            "budget_2024_zmw": amounts[-2] if len(amounts) >= 2 else None,
            "budget_2025_zmw": amounts[-1] if len(amounts) >= 1 else None,
        })

df_2025 = pd.DataFrame(rows)
parent_total_2025 = df_2025[df_2025['level']==1]['budget_2025_zmw'].sum()

print(f"Total rows: {len(df_2025)}")
print(f"Parent programmes: {(df_2025['level'] == 1).sum()}")
print(f"Sub-programmes:    {(df_2025['level'] == 2).sum()}")
print(f"\n2025 parent total: {parent_total_2025:,} ZMW  (expected ≈ 223,603,854)")
print(f"Difference: {parent_total_2025 - 223603854:+,} ZMW")
print()
print(df_2025[df_2025['level'] == 1][['code', 'programme', 'budget_2025_zmw']].to_string(index=False))

Total rows: 59
Parent programmes: 14
Sub-programmes:    45

2025 parent total: 222,580,695.0 ZMW  (expected ≈ 223,603,854)
Difference: -1,023,159.0 ZMW

code                                   programme  budget_2025_zmw
   1                    Constituency Development       36058150.0
   2                            Local Governance       12077969.0
   3             Integrated Development Planning        5339802.0
   4           Economic and business development        2565410.0
   5  Public health and Environmental protection       17345043.0
   6             Housing and Community Amenities       34266632.0
   7             Recreation Culture and Religion        4274621.0
   8            Education and skills development        2248988.0
  10                     Public order and safety       18684734.0
  11             Management and support Services       60515493.0
  12        Resource Mobilisation and Management       13120771.0
  13                    District Health servcies       

In [17]:
import pandas as pd

# Ensure all columns exist
for c in ["budget_2023_zmw", "budget_2024_zmw", "budget_2025_zmw"]:
    if c not in df_2025.columns:
        df_2025[c] = None

# Reorder
df_2025 = df_2025[["level", "code", "programme", "sub_programme",
                   "budget_2023_zmw", "budget_2024_zmw", "budget_2025_zmw"]]

df_2025.to_csv(
    "db-unza26-csc4792-chililabombwe_budget_programmes_2025.csv",
    sep="|",
    index=False
)
print("✅ Saved: db-unza26-csc4792-chililabombwe_budget_programmes_2025.csv")
print(f"Rows: {len(df_2025)}")
print(f"2025 total: K{df_2025[df_2025['level']==1]['budget_2025_zmw'].sum():,.2f}")

✅ Saved: db-unza26-csc4792-chililabombwe_budget_programmes_2025.csv
Rows: 59
2025 total: K222,580,695.00


In [18]:
import pdfplumber
import pandas as pd
import re

BUDGET_2025 = "pdfs/2025-CHILILABOMBWE-MUNICIPAL-COUNCIL-ADJUSTED-BUDGET.pdf"

# Extract raw text from page 14
with pdfplumber.open(BUDGET_2025) as pdf:
    text = pdf.pages[13].extract_text() or ""

# The indicators and targets are on lines that end with numbers
lines = text.split("\n")

targets = []
for line in lines:
    line = line.strip()
    # Match lines like "01 Number of school desks procured (0) - - (0) 600"
    m = re.match(r'^(\d{2})\s+(.+)', line)
    if m:
        indicator = m.group(2)
        # Find all tokens in the line
        numbers = re.findall(r'[\d,]+', indicator)
        # Last number is 2025 Target
        if numbers:
            target = numbers[-1].replace(",", "")
            try:
                target_val = int(target)
            except:
                target_val = None
            # Clean indicator name — remove numbers and parens
            clean_name = re.sub(r'\(0\)|\(\d+\)|-\s*', '', indicator)
            clean_name = re.sub(r'[\d,]+', '', clean_name).strip()
            targets.append({
                "indicator_code": m.group(1),
                "indicator": clean_name,
                "target_2025": target_val,
            })

df_targets = pd.DataFrame(targets)
print(f"Extracted {len(df_targets)} output indicators:")
print(df_targets.to_string(index=False))

Extracted 11 output indicators:
indicator_code                                                        indicator  target_2025
            01                                  Number of school desks procured          600
            01                          Number of maternity annexes constructed            2
            01                       Kilometres of township roads rehabilitated           50
            01                           Number of * classroom blocks completed            3
            01                                      Number of boreholes drilled            3
            01 Number of youth women and community groups empowered with grants          160
            01   Number of youthyomen and community groups empowered with loans          100
            01                                 Number of CDF projects monitored            2
            02                                  Number of CDF pojects appraised            2
            01       Numberof secondar

In [19]:
if len(df_targets) > 0:
    df_targets.to_csv(
        "db-unza26-csc4792-chililabombwe_output_targets_2025.csv",
        sep="|",
        index=False
    )
    print("✅ Saved: db-unza26-csc4792-chililabombwe_output_targets_2025.csv")

✅ Saved: db-unza26-csc4792-chililabombwe_output_targets_2025.csv


In [20]:
import pdfplumber
import pandas as pd
import re

with pdfplumber.open(BUDGET_2025) as pdf:
    text = pdf.pages[8].extract_text() or ""

# Parse lines like "(1) GRZ - CDF Grant 18,684,678"
lines = text.split("\n")
revenue = []
current_source = None

for line in lines:
    line = line.strip()
    m = re.match(r'^\((\d+)\)\s*(GRZ[^\d]+?)\s+([\d,]+)', line)
    if m:
        revenue.append({
            "line_id": m.group(1),
            "source": m.group(2).strip(),
            "amount_2025_zmw": int(m.group(3).replace(",", "")),
        })
    else:
        # Lines without a leading number but with source+amount
        m2 = re.match(r'^(GRZ[^\d]+?)\s+([\d,]+)', line)
        if m2:
            revenue.append({
                "line_id": None,
                "source": m2.group(1).strip(),
                "amount_2025_zmw": int(m2.group(2).replace(",", "")),
            })

df_revenue = pd.DataFrame(revenue)
print(f"Extracted {len(df_revenue)} revenue lines")
print(df_revenue.to_string(index=False))

if len(df_revenue) > 0:
    df_revenue.to_csv(
        "db-unza26-csc4792-chililabombwe_revenue_sources_2025.csv",
        sep="|",
        index=False
    )
    print("✅ Saved: db-unza26-csc4792-chililabombwe_revenue_sources_2025.csv")

Extracted 23 revenue lines
line_id             source  amount_2025_zmw
   None    GRZ - CDF Grant         18684678
   None    GRZ - CDF Grant          2491290
   None    GRZ - CDF Grant          1522925
   None    GRZ - CDF Grant          6228226
   None   GRZ - LGEF Grant          1317940
   None   GRZ - LGEF Grant           177989
   None   GRZ - LGEF Grant           585413
   None   GRZ - LGEF Grant            94780
   None   GRZ - LGEF Grant          3949368
   None   GRZ - LGEF Grant           915305
   None   GRZ - LGEF Grant             1439
   None   GRZ - LGEF Grant           404780
   None   GRZ - LGEF Grant            11350
   None   GRZ - LGEF Grant             2613
   None   GRZ - LGEF Grant          4948815
   None GRZ - LGEF Local R            56269
   None   GRZ - LGEF Grant          1109854
   None   GRZ - LGEF Grant           402327
   None   GRZ - LGEF Grant           313432
   None   GRZ - LGEF Grant           757487
   None   GRZ - LGEF Grant         12570771
   No

In [21]:
import os

csvs = sorted([f for f in os.listdir(".") if f.endswith(".csv")])
print(f"CSVs in current folder ({len(csvs)}):\n")
for f in csvs:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {size_kb:7.1f} KB  {f}")

CSVs in current folder (6):

      0.1 KB  db-unza26-csc4792-chililabombwe_budget_comparison_2023_2024.csv
      4.4 KB  db-unza26-csc4792-chililabombwe_budget_programmes_2025.csv
      1.4 KB  db-unza26-csc4792-chililabombwe_cdf_projects_2024.csv
      3.4 KB  db-unza26-csc4792-chililabombwe_obb_budget_2024.csv
      0.6 KB  db-unza26-csc4792-chililabombwe_output_targets_2025.csv
      0.6 KB  db-unza26-csc4792-chililabombwe_revenue_sources_2025.csv
